In [2]:
import sys
sys.path.append("..")

In [3]:
import pandas as pd
from item import Item
from ASRSManager import ASRSManager


manager = ASRSManager(config_path='../config.yaml')

# ===============================================================
# 1: Online Operation, with nearest bin first fit algorithm
# ===============================================================
print ("=====online first fit algorithm=====")
item_list = []
df = pd.read_csv("../items.csv")
for row in df.itertuples(index=False):
    item_list.append(Item(
        width=row.width, 
        height=row.height, 
        depth=row.depth, 
        rotation=row.can_rotate, 
        weight=row.weight, 
        cargo_id=row.cargo_id,
        empty=False
    ))

# start to place items online
for item in item_list:
    result = manager.place_item_online(item)
    
    if not result:
        print(f"failed to place item {item.cargo_id} online.")

item_position_online = {}
total_length_online = 0
vertical_distance = 0
horizontal_distance = 0
for item in item_list:
    retrieved_item = manager.retrieve_item(cargo_id=item.cargo_id)
    if retrieved_item:
        item_position_online[item.cargo_id] = (item.placed_bin, retrieved_item["position"])
        vertical_distance += abs(retrieved_item["position"][1] - manager.entrance_position[1])
        horizontal_distance += manager.bin_dimensions[0] * abs(int(retrieved_item["placed_bin"]) - manager.entrance_position[3])

total_length_online += vertical_distance + horizontal_distance

print (f"Total horizontal distance of items placed online: {horizontal_distance}")
print (f"Total vertical distance of items placed online: {vertical_distance}")
print(f"Total length of items placed online: {total_length_online}")
print("\n")

# ===============================================================
# 2: Offline Reorganization
# ===============================================================
print ("=====offline reorganization=====")
reorg_result = manager.reorganize_offline()

total_length_offline = 0
vertical_distance = 0
horizontal_distance = 0
if reorg_result:
    print(f"reorganization successful!")
    for item in item_list:
        retrieved_item = manager.retrieve_item(cargo_id=item.cargo_id)
        if retrieved_item is not None:
            vertical_distance += abs(retrieved_item["position"][1] - manager.entrance_position[1])
            horizontal_distance += manager.bin_dimensions[0] * abs(int(retrieved_item["placed_bin"]) - manager.entrance_position[3])

    total_length_offline += vertical_distance + horizontal_distance
    print(f"Total horizontal distance of items moved offline: {horizontal_distance}")
    print(f"Total vertical distance of items moved offline: {vertical_distance}")
    print (f"Total length of moving items offline: {total_length_offline}")
else:
    print(f"reorganization failed.")

print("\n")

# ===============================================================
# 4: Offline Reorganization, without nearest bin first fit algorithm
# ===============================================================
print ("=====offline reorganization with reverse sorting by height=====")
reorg_result = manager.exp_reorganize_offline()

total_length_offline = 0
vertical_distance = 0
horizontal_distance = 0
if reorg_result:
    print(f"reorganization successful!")
    for item in item_list:
        retrieved_item = manager.retrieve_item(cargo_id=item.cargo_id)
        if retrieved_item is not None:
            vertical_distance += abs(retrieved_item["position"][1] - manager.entrance_position[1])
            horizontal_distance += manager.bin_dimensions[0] * abs(int(retrieved_item["placed_bin"]) - manager.entrance_position[3])

    total_length_offline += vertical_distance + horizontal_distance
    print(f"Total horizontal distance of items moved offline: {horizontal_distance}")
    print(f"Total vertical distance of items moved offline: {vertical_distance}")
    print (f"Total length of moving items offline: {total_length_offline}")
else:
    print(f"reorganization failed.")

=====online first fit algorithm=====
Total horizontal distance of items placed online: 5800
Total vertical distance of items placed online: 4530
Total length of items placed online: 10330


=====offline reorganization=====
reorganization successful!
Total horizontal distance of items moved offline: 8450
Total vertical distance of items moved offline: 4495
Total length of moving items offline: 12945


=====offline reorganization with reverse sorting by height=====
reorganization successful!
Total horizontal distance of items moved offline: 9700
Total vertical distance of items moved offline: 4210
Total length of moving items offline: 13910


In [4]:
## ===============================================================
# 3: Online Operation, with random bin priority
# ===============================================================

print ("=====online first fit algorithm with random bin priority=====")

num_iters = 50
average_horizontal_distance = 0
average_vertical_distance = 0
total_length_online = 0
for i in range(num_iters):
    manager2 = ASRSManager(
        config_path='../config.yaml'
    )
    item_list = []
    df = pd.read_csv("../items.csv")
    for row in df.itertuples(index=False):
        item_list.append(Item(
            width=row.width, 
            height=row.height, 
            depth=row.depth, 
            rotation=row.can_rotate, 
            weight=row.weight, 
            cargo_id=row.cargo_id,
            empty=False
        ))
    import random
    random.shuffle(manager2.online_priority)
    for item in item_list:
        result = manager2.place_item_online(item)
        
        if not result:
            print(f"failed to place item {item.cargo_id} online.")
    
    item_position_online = {}
    # vertical_distance = 0
    # horizontal_distance = 0
    for item in item_list:
        retrieved_item = manager2.retrieve_item(cargo_id=item.cargo_id)
        if retrieved_item:
            item_position_online[item.cargo_id] = (item.placed_bin, retrieved_item["position"])
            average_vertical_distance += abs(retrieved_item["position"][1] - manager.entrance_position[1])
            average_horizontal_distance += manager2.bin_dimensions[0] * abs(int(retrieved_item["placed_bin"]) - manager2.entrance_position[3])

    total_length_online += vertical_distance + horizontal_distance
print ("total length online = ", total_length_online)
print ("average length online = ", total_length_online / num_iters)
print ("average horizontal distance online = ", average_horizontal_distance / num_iters)
print ("average vertical distance online = ", average_vertical_distance / num_iters)

# for bin in manager2.bins.values():
#     print ("\n")
#     print (f"items in bin {bin.id}: ")
#     for item in bin.items.values():
#         print (f"id:{item.pallet_id}, height:{item.height}")


=====online first fit algorithm with random bin priority=====
total length online =  695500
average length online =  13910.0
average horizontal distance online =  7585.0
average vertical distance online =  4530.0


In [6]:
import subprocess

num_iters = 50
original_vertical = 0
original_horizontal = 0
exp_vertical = 0
exp_horizontal = 0

for i in range(num_iters):
    subprocess.run(["python3", "random_item.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    manager = ASRSManager(config_path='../config.yaml')

    item_list = []
    df = pd.read_csv("../items.csv")
    for row in df.itertuples(index=False):
        item_list.append(Item(
            width=row.width, 
            height=row.height, 
            depth=row.depth, 
            rotation=row.can_rotate, 
            weight=row.weight, 
            cargo_id=row.cargo_id,
            empty=False
        ))

    # start to place items online
    for item in item_list:
        result = manager.place_item_online(item)
        
        if not result:
            print(f"failed to place item {item.cargo_id} online.")

    reorg_result = manager.reorganize_offline()

    if reorg_result:
        for item in item_list:
            retrieved_item = manager.retrieve_item(cargo_id=item.cargo_id)
            if retrieved_item is not None:
                original_vertical += abs(retrieved_item["position"][1] - manager.entrance_position[1])
                original_horizontal += manager.bin_dimensions[0] * abs(int(retrieved_item["placed_bin"]) - manager.entrance_position[3])
    else:
        print(f"reorganization failed.")


    # ===============================================================
    # 4: Offline Reorganization, without nearest bin first fit algorithm
    # ===============================================================
    reorg_result = manager.exp_reorganize_offline()

    if reorg_result:
        for item in item_list:
            retrieved_item = manager.retrieve_item(cargo_id=item.cargo_id)
            if retrieved_item is not None:
                exp_vertical += abs(retrieved_item["position"][1] - manager.entrance_position[1])
                exp_horizontal += manager.bin_dimensions[0] * abs(int(retrieved_item["placed_bin"]) - manager.entrance_position[3])
    else:
        print(f"reorganization failed.")


print (f"Average original vertical distance: {original_vertical / num_iters}")
print (f"Average original horizontal distance: {original_horizontal / num_iters}")
print (f"Average original total distance: {(original_vertical + original_horizontal) / num_iters}")
print (f"Average experimental vertical distance: {exp_vertical / num_iters}")
print (f"Average experimental horizontal distance: {exp_horizontal / num_iters}")
print (f"Average experimental total distance: {(exp_vertical + exp_horizontal) / num_iters}")

Average original vertical distance: 4495.0
Average original horizontal distance: 8450.0
Average original total distance: 12945.0
Average experimental vertical distance: 4210.0
Average experimental horizontal distance: 9700.0
Average experimental total distance: 13910.0
